### Importing libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Models
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor


### Reading Data

In [5]:
data=pd.read_csv('data/StudentsPerformance.csv')
data.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


### Preparing X and Y Variables

In [19]:
# axis=0 (rows)
# Think:
# “I am removing data points”
# axis=1 (columns)
# Think:
# “I am removing features”

X=data.drop(columns=['math score'])
y=data['math score']
X.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [ ]:
# Creating Column Transformer which converts the transformation in a pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_feature=X.select_dtypes(exclude='object').columns
cat_features=X.select_dtypes(include='object').columns

num_transformer=StandardScaler()
cat_transformer=OneHotEncoder()

preprocessor=ColumnTransformer(
    [
        ('OneHotEncoder', cat_transformer, cat_features),
        ('StandardScaler', num_transformer, num_feature)
    ]
)   

In [ ]:
X=preprocessor.fit_transform(X)

### Separate DataSet into Training data nd Test Data

In [35]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.25, random_state=42)
X_train.shape, X_test.shape

((750, 19), (250, 19))

### Function to give all metrics for the model after training

In [36]:
def evaluate(true, predicted):
    mse=mean_squared_error(true,predicted)
    mae=mean_absolute_error(true,predicted)
    rmse=np.sqrt(mse)
    r2_sc=r2_score(true,predicted)
    return mae, rmse, r2_sc

In [37]:
print(type(X_train))
print(X_train.shape)

print(type(X_train[0]))

<class 'numpy.ndarray'>
(750, 19)
<class 'numpy.ndarray'>


In [43]:
# Creating all Models
models={
    'Linear Regression':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'K-Nearest Neighbor Regressor':KNeighborsRegressor(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'XG Boost Regressor':XGBRegressor(),
    'Cat Boost Regressor':CatBoostRegressor(),
    'AdaBoost Regressor':AdaBoostRegressor()
}
model_list=[]
r2_list=[]
rmse_list=[]

for name, model in models.items():
    model.fit(X_train, y_train)  # Model Training

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate
    model_train_mae, model_train_rmse, model_train_r2Score = evaluate(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2Score = evaluate(y_test, y_test_pred)

    print(name)
    model_list.append(model)
    r2_list.append(model_test_r2Score)
    rmse_list.append(model_test_rmse)

    print("Model performance for Training set")
    print(f"- Root Mean Squared Error: {model_train_rmse:.4f}")
    print(f"- Mean Absolute Error: {model_train_mae:.4f}")
    print(f"- R2 Score: {model_train_r2Score:.4f}")

    print("-" * 30)

    print("Model performance for Test set")
    print(f"- Root Mean Squared Error: {model_test_rmse:.4f}")
    print(f"- Mean Absolute Error: {model_test_mae:.4f}")
    print(f"- R2 Score: {model_test_r2Score:.4f}")

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.2972
- Mean Absolute Error: 4.2383
- R2 Score: 0.8743
------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.4825
- Mean Absolute Error: 4.3379
- R2 Score: 0.8778
Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5515
- Mean Absolute Error: 5.1837
- R2 Score: 0.8077
------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.6541
- Mean Absolute Error: 5.2217
- R2 Score: 0.8200
Ridge
Model performance for Training set
- Root Mean Squared Error: 5.2976
- Mean Absolute Error: 4.2368
- R2 Score: 0.8743
------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.4788
- Mean Absolute Error: 4.3354
- R2 Score: 0.8780
K-Nearest Neighbor Regressor
Model performance for Training set
- Root Mean Squared Error: 5.7920
- Mean Absolute Error: 4.5864
- R2 Score: 0.8497
------------------------------
Mod

## Results

In [46]:
res=pd.DataFrame(list(zip(model_list,r2_list,rmse_list)), columns=['Model Name','R2 Score', 'RMSE']).sort_values(['R2 Score'], ascending=False)
res

,Model Name,R2 Score,RMSE
2,Ridge(),0.877990,5.478806
0,LinearRegression(),0.877824,5.482528
7,CatBoostRegressor(loss_function='RMSE'),0.854830,5.976235
5,"(DecisionTreeRegressor(max_features=1.0, rando...",0.848376,6.107626
8,"(DecisionTreeRegressor(max_depth=3, random_sta...",0.844528,6.184641
6,"XGBRegressor(base_score=None, booster=None, ca...",0.836290,6.346378
1,Lasso(),0.820027,6.654136
3,KNeighborsRegressor(),0.793207,7.132741
4,DecisionTreeRegressor(),0.758901,7.701688
